# First detection of researchers using spaCy

In [1]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


## Process groups and centers JSON files and extract people

In [2]:
import json
from pathlib import Path

# Load json files
def load_json(filepath):
    with Path(filepath).open(encoding="utf-8") as f:
        return json.load(f)


def clean_string(text):
    return " ".join((text or "").split())


groups = load_json("../json/grupos_detalle.json")
centers = load_json("../json/centros_detalle.json")

people = set()

# Groups
for group in groups:
    for miembro in group.get("miembros", []):
        nombre = clean_string(miembro.get("nombre"))
        if nombre:
            people.add(nombre)

    for colaborador in group.get("colaboradores", []):
        nombre = clean_string(colaborador.get("nombre"))
        if nombre:
            people.add(nombre)

# Centers
for center in centers:
    for investigador in center.get("investigadores", []):
        nombre = clean_string(investigador.get("nombre"))
        if nombre:
            people.add(nombre)

print(f"People in groups and centers: {len(people)}")
print("Groups:", len(groups))
print("Centers:", len(centers))

People in groups and centers: 2284
Groups: 222
Centers: 26


### Add independent researchers to the list of people

In [3]:
# Researchers without group or center
all_researchers = load_json("../json/todos_investigadores.json")

for researcher in all_researchers:
    nombre = clean_string(researcher.get("nombre").strip() + " " + (researcher.get("apellido1") or "").strip() + " " + (researcher.get("apellido2") or "").strip())
    if nombre:
        people.add(nombre)

# Convertir a lista ordenada
person_strings = sorted(people)

print(f"Unique people: {len(person_strings)}")

Unique people: 11996


### Add combination of surnames to the list of people, improving matcher capacity

In [4]:

# Researchers without second surname
for researcher in all_researchers:
    nombre = clean_string(researcher.get("nombre").strip() + " " + (researcher.get("apellido1") or "").strip()).strip()
    if nombre:
        people.add(nombre)

print(f"Unique people (add combinations without second surname): {len(person_strings)}")

# Researchers with composed names (e.g., María Pilar García Cuetos)
name_connectors = {"de", "del", "da", "das", "do", "dos", "la", "las", "los", "van", "von", "y", "a", "al"}
for researcher in all_researchers:
    # Check for composed names (only with two names)
    nombre_splitted = (researcher.get("nombre") or "").strip().split(" ")
    if researcher.get("nombre") and " " in researcher.get("nombre") and len(nombre_splitted) == 2:
        # Always create variants from the first part of the name
        nombre = clean_string(nombre_splitted[0] + " " + (researcher.get("apellido1") or "").strip() + " " + (researcher.get("apellido2") or "").strip())
        if nombre:
            people.add(nombre)
        nombre = clean_string(nombre_splitted[0] + " " + (researcher.get("apellido1") or "").strip())
        if nombre:
            people.add(nombre)

        # Only use the second part if it is a real given name, not a connector like 'del'
        if nombre_splitted[1].lower() not in name_connectors:
            nombre = clean_string(nombre_splitted[1] + " " + (researcher.get("apellido1") or "").strip() + " " + (researcher.get("apellido2") or "").strip())
            if nombre:
                people.add(nombre)
            nombre = clean_string(nombre_splitted[1] + " " + (researcher.get("apellido1") or "").strip())
            if nombre:
                people.add(nombre)

person_strings = sorted(people)

print(f"Unique people (add combinations with composed names): {len(person_strings)}")

Unique people (add combinations without second surname): 11996
Unique people (add combinations with composed names): 33294


### Add governing authorities

In [5]:
# University goverment
governing_board_members = load_json("../json/gobierno_universidad.json")
governing_board_people = set()

for member in governing_board_members:
    name = clean_string(member.get("nombre").strip() + " " + (member.get("apellido1") or "").strip() + " " + (member.get("apellido2") or "").strip())
    name_with_one_surname = clean_string(member.get("nombre").strip() + " " + (member.get("apellido1") or "").strip())
    if name:
        governing_board_people.add(name)
    if name_with_one_surname:
        governing_board_people.add(name_with_one_surname)


# Province government
province_government_members = load_json("../json/gobierno_principado.json")
province_government_people = set()

for member in province_government_members:
    name = clean_string(member.get("nombre").strip() + " " + (member.get("apellido1") or "").strip() + " " + (member.get("apellido2") or "").strip())
    name_with_one_surname = clean_string(member.get("nombre").strip() + " " + (member.get("apellido1") or "").strip())
    if name:
        province_government_people.add(name)
    if name_with_one_surname:
        province_government_people.add(name_with_one_surname)


# National government
national_government_members = load_json("../json/gobierno_nacional.json")
national_government_people = set()

for member in national_government_members:
    name = clean_string(member.get("nombre").strip() + " " + (member.get("apellido1") or "").strip() + " " + (member.get("apellido2") or "").strip())
    name_with_one_surname = clean_string(member.get("nombre").strip() + " " + (member.get("apellido1") or "").strip())
    if name:
        national_government_people.add(name)
    if name_with_one_surname:
        national_government_people.add(name_with_one_surname)

## Add matchers

In [ ]:
import unicodedata

import spacy
from spacy.matcher import PhraseMatcher

nlp = spacy.blank("es")

_TRANSLATION_TABLE = str.maketrans({
    "‘": "'",
    "’": "'",
    "“": '"',
    "”": '"',
    "–": "-",
    "—": "-",
    "\xa0": " ",
})

def normalize_text(text):
    text = text.translate(_TRANSLATION_TABLE)
    return "".join(
        char
        for char in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(char)
    )


def normalized_patterns(values: list[str]):
    normalized_values = sorted(
        {
            normalize_text(clean_string(value))
            for value in values
            if clean_string(value)
        }
    )
    print(normalized_values)
    return list(nlp.pipe(normalized_values))


# Without attr="LOWER" (al ser acrónimos y nombres se deben respetar las mayúsculas) -----------------
# Centers matcher
center_matcher = PhraseMatcher(nlp.vocab)
center_acronym_patterns = normalized_patterns(center.get("acronimo") for center in centers)
center_matcher.add("CENTER_ACRONYM", center_acronym_patterns)
center_name_patterns = normalized_patterns(center.get("nombre_centro") for center in centers)
center_name_patterns += normalized_patterns(center.get("alt_nombre_centro") for center in centers)
center_matcher.add("CENTER_NAME", center_name_patterns)

# Group matcher
group_matcher = PhraseMatcher(nlp.vocab)
group_patterns = normalized_patterns(group.get("grupo_acronimo") for group in groups)
group_matcher.add("GROUP_ACRONYM", group_patterns)
group_name_patterns = normalized_patterns(group.get("nombre_grupo") for group in groups)
group_matcher.add("GROUP_NAME", group_name_patterns)


# With attr="LOWER" -----------------
# Researchers matchers
researchers_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
researchers_patterns = normalized_patterns(person_strings)
researchers_matcher.add("RESEARCHER", researchers_patterns)

# Governing board matcher
university_government_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
university_government_patterns = normalized_patterns(governing_board_people)
university_government_matcher.add("UNIVERSITY_GOVERNMENT_MEMBER", university_government_patterns)


# Province government matcher
province_government_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
province_government_patterns = normalized_patterns(province_government_people)
province_government_matcher.add("PROVINCE_GOVERNMENT_MEMBER", province_government_patterns)


# National government matcher
national_government_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
national_government_patterns = normalized_patterns(national_government_people)
national_government_matcher.add("NATIONAL_GOVERNMENT_MEMBER", national_government_patterns)



['AsRaM', 'BME', 'C1NN', 'CEISIA', 'CINN', 'CUIDA', 'CeCodet', 'ICTEA', 'IFID', 'INDUROT', 'INEUROPA', 'IUBA', 'IUDE', 'IUFV', 'IUGENDIV', 'IUOPA', 'IUQOEM', 'IUTA']
['Aula Valdes Salas', 'Catedra Cohen', 'Centro Mixto de Investigacion en Nanomateriales y Nanotecnologia', 'Centro Universitario de Investigacion Asturias Raw Materials', 'Centro Universitario de Investigacion y Desarrollo del Agua', 'Centro de Cooperacion y Desarrollo Territorial', 'Centro de Estudios sobre el Impacto Social de la Inteligencia Artificial', 'Centro de Ingenieria Biomedica', 'Centro de Innovacion', 'Centro de Servicios Universitarios de Aviles', "Escuela Internacional de Doctorado 'Paz Suarez Rendueles'", 'Instituto Feijoo de Estudios del Siglo XVIII', 'Instituto Mixto de Investigacion en Biodiversidad', 'Instituto Universitario Fernandez Vega', 'Instituto Universitario de Biotecnologia de Asturias', 'Instituto Universitario de Oncologia', 'Instituto Universitario de Quimica Organometalica «Enrique Moles»',

## Apply matcher to news articles

In [7]:
input_path = Path("../json/noticias_uniovi.json")
output_path = Path("./results_phrase_matcher_noticias_uniovi.json")

def extract_entities(text):
    text = text or ""
    normalized_text = normalize_text(text)
    doc = nlp(normalized_text)

    matches = list(researchers_matcher(doc))
    matches += list(center_matcher(doc))
    matches += list(group_matcher(doc))
    matches += list(university_government_matcher(doc))
    matches += list(province_government_matcher(doc))
    matches += list(national_government_matcher(doc))
    matches = sorted(matches, key=lambda match: (-(match[2] - match[1]), match[1]))

    entities = []
    for match_id, token_start, token_end in matches:
        span = doc[token_start:token_end]
        start_char = span.start_char
        end_char = span.end_char
        label = nlp.vocab.strings[match_id]

        # If this entity is contained in a longer accepted one, skip it regardless of label.
        if any(
            existing["start"] <= start_char
            and existing["end"] >= end_char
            and (existing["start"] < start_char or existing["end"] > end_char)
            for existing in entities
        ):
            continue

        # Avoid exact duplicates (same span + same label).
        if any(
            existing["start"] == start_char
            and existing["end"] == end_char
            and existing["label"] == label
            for existing in entities
        ):
            continue

        # Remove shorter accepted entities contained by this one.
        entities = [
            existing
            for existing in entities
            if not (
                start_char <= existing["start"]
                and end_char >= existing["end"]
                and (start_char < existing["start"] or end_char > existing["end"])
            )
        ]

        entities.append(
            {
                "start": start_char,
                "end": end_char,
                "token_start": token_start,
                "token_end": token_end,
                "label": label,
                "text": text[start_char:end_char],
                "normalized_text": span.text,
            }
        )

    return entities


with input_path.open(encoding="utf-8") as f:
    news = json.load(f)

news_with_ner = []
for item in news:
    enriched_item = dict(item)
    enriched_item["ner_titulo"] = extract_entities(item.get("titulo", ""))
    enriched_item["ner_resumen"] = extract_entities(item.get("resumen", ""))
    enriched_item["ner_texto"] = extract_entities(item.get("noticia", ""))
    news_with_ner.append(enriched_item)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(news_with_ner, f, ensure_ascii=False, indent=2)

print(f"Saved {len(news_with_ner)} articles to {output_path}")

Saved 303 articles to results_phrase_matcher_noticias_uniovi.json


In [8]:
import spacy
from spacy.matcher import Matcher

# Download en_core_web_sm
!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)
# Add match ID "HelloWorld" with no callback and one pattern
pattern = [{"LOWER": "hello"}, {"IS_PUNCT": True}, {"LOWER": "world"}]
matcher.add("HelloWorld", [pattern])

doc = nlp("Hello, world! Hello world!")
matches = matcher(doc)
for match_id, start, end in matches:
    string_id = nlp.vocab.strings[match_id]  # Get string representation
    span = doc[start:end]  # The matched span
    print(match_id, string_id, start, end, span.text)

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 5.2 MB/s eta 0:00:03
     ---- ----------------------------------- 1.6/12.8 MB 5.0 MB/s eta 0:00:03
     -------- ------------------------------- 2.6/12.8 MB 5.2 MB/s eta 0:00:02
     ------------ --------------------------- 3.9/12.8 MB 5.2 MB/s eta 0:00:02
     --------------- ------------------------ 5.0/12.8 MB 5.3 MB/s eta 0:00:02
     ------------------- -------------------- 6.3/12.8 MB 5.4 MB/s eta 0:00:02
     ---------------------- ----------------- 7.3/12.8 MB 5.4 MB/s eta 0:00:02
     --------------------------- ------------ 8.9/12.8 MB 5.6 MB/s eta 0:00:01
     ------------------------------- -------- 10.0/12.8 MB 5.6 MB/s eta 0:00:01
     ------------------------------------ --- 11.5/12.8 MB 5.7 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 5.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 

In [9]:
from spacy.lang.en import English
from spacy.matcher import PhraseMatcher

nlp = English()
matcher = PhraseMatcher(nlp.vocab, attr="SHAPE")
matcher.add("IP", [nlp("127.0.0.1"), nlp("127.127.0.0")])

doc = nlp("Often the router will have an IP address such as 192.168.1.1 or 192.168.2.1.")
for match_id, start, end in matcher(doc):
    print("Matched based on token shape:", doc[start:end])

Matched based on token shape: 192.168.1.1
Matched based on token shape: 192.168.2.1
